# Débruitage audio par seuillage temps-fréquence

On observe un signal audio $\mathbf{y} = \mathbf{x} + \mathbf{n}$, où $\mathbf{x}$ est le signal propre et $\mathbf{n}$ un bruit blanc gaussien d'écart-type $\sigma$. On cherche à estimer $\mathbf{x}$ à partir de $\mathbf{y}$ seul.

Les débruiteurs de ce notebook ont tous la même forme :

$$\hat{\mathbf{x}} = \boldsymbol\Phi\, g(\boldsymbol\Phi^\ast \mathbf{y}).$$

- $\boldsymbol\Phi^\ast$ est l'analyse : la transformée de Fourier à court terme (STFT) du signal.
- $g$ modifie les coefficients $\mathbf{Y} = \boldsymbol\Phi^\ast\mathbf{y}$ : il garde les grands, atténue les petits.
- $\boldsymbol\Phi$ est la synthèse : elle ramène les coefficients modifiés dans le domaine temporel.

Plan :

1. le signal et le bruit ;
2. la STFT : analyse, synthèse, inverse ;
3. les seuillages ponctuels (doux, dur, Wiener empirique) et le niveau $\lambda \in [0, 1]$ ;
4. un seuillage social : le Persistent Empirical Wiener (PEW) ;
5. le block thresholding de Yu, Mallat et Bacry [1] ;
6. le post-traitement de Wiener.

Les deux modules utilisés sont dans ce répertoire : `stft.py` et `tfdenoise.py`.

## Préparation

Le calcul se fait sur le premier périphérique disponible : CUDA, puis MPS (Apple), puis CPU.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from stft import STFT
import tfdenoise as tfd

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("périphérique :", device)

## 1. Le signal et le bruit

**Paramètres.** `isnr_db` fixe le niveau de bruit, en décibels. `seed` fixe le tirage du bruit.

In [ ]:
isnr_db = 10.0   # rapport signal sur bruit d'entrée, en dB
seed = 0         # graine du bruit
fs = 44_100      # fréquence d'échantillonnage, en Hz

Le signal par défaut est un signal jouet de 1,5 s : une note à huit partiels, un chirp linéaire et trois frappes percussives. Pour prendre un extrait de fichier WAV (16 bits, 44,1 kHz), donner son chemin dans `wav_path`.

In [ ]:
def toy_signal(n=512 * 129, fs=44_100):
    # Note à huit partiels, chirp linéaire, trois frappes percussives.
    t = np.arange(n, dtype=np.float32) / fs
    rng = np.random.default_rng(7)
    x = np.zeros(n, dtype=np.float32)
    phase = np.cumsum(2 * np.pi / fs * 220 * (1 + 0.01 * np.sin(2 * np.pi * 5 * t)))
    env = np.clip((t - 0.10) / 0.02, 0, 1) * np.exp(-(t - 0.10) / 1.0) * (t < 1.25)
    for p in range(1, 9):
        x += env * (0.5 / p) * np.sin(p * phase)
    tc = t - 0.2
    x += 0.08 * ((t >= 0.2) & (t < 1.4)) * np.sin(2 * np.pi * (600 * tc + 0.5 * (4400 / 1.2) * tc ** 2))
    for t0 in (0.30, 0.75, 1.20):
        x += 0.5 * (t >= t0) * np.exp(-np.clip(t - t0, 0, None) / 0.01) * rng.standard_normal(n)
    return x.astype(np.float32)


def load_wav(path, start_s=30.0, duration_s=5.0):
    # Extrait mono d'un WAV PCM 16 bits, en float32 dans [-1, 1].
    import wave
    with wave.open(str(path)) as w:
        assert w.getsampwidth() == 2, "WAV 16 bits attendu"
        rate, channels = w.getframerate(), w.getnchannels()
        w.setpos(int(start_s * rate))
        raw = w.readframes(int(duration_s * rate))
    x = np.frombuffer(raw, dtype=np.int16).reshape(-1, channels).mean(axis=1)
    return (x / 32768.0).astype(np.float32), rate


wav_path = None   # par exemple "mon_morceau.wav"

if wav_path is None:
    x_np = toy_signal(fs=fs)
else:
    x_np, fs = load_wav(wav_path)
x = torch.from_numpy(x_np).to(device)
N = x.numel()
print(f"N = {N} échantillons, {N / fs:.2f} s")
display(Audio(x_np, rate=fs))

Le rapport signal sur bruit d'une estimée $\hat{\mathbf{x}}$ est

$$\mathrm{SNR}(\mathbf{x}, \hat{\mathbf{x}}) = 10 \log_{10} \frac{\lVert\mathbf{x}\rVert^2}{\lVert\mathbf{x} - \hat{\mathbf{x}}\rVert^2}.$$

L'iSNR est le SNR de l'observation, $\hat{\mathbf{x}} = \mathbf{y}$. L'oSNR est celui du signal débruité.

Pour un bruit blanc, $\mathbb{E}\lVert\mathbf{n}\rVert^2 = N\sigma^2$. On règle donc $\sigma = \frac{\lVert\mathbf{x}\rVert}{\sqrt N}\, 10^{-\mathrm{iSNR}/20}$.

In [ ]:
def snr_db(ref, est):
    return 10 * torch.log10(ref.pow(2).sum() / (ref - est).pow(2).sum()).item()


sigma = x.pow(2).mean().sqrt().item() * 10 ** (-isnr_db / 20)
noise = sigma * torch.randn(N, generator=torch.Generator().manual_seed(seed))
y = x + noise.to(device)
print(f"σ = {sigma:.4f}, iSNR mesuré = {snr_db(x, y):.2f} dB")
display(Audio(y.cpu().numpy(), rate=fs))

## 2. La STFT

La STFT découpe le signal en trames de $N_w$ échantillons, espacées de $H$, multipliées par une fenêtre $w$, puis en prend la transformée de Fourier. Les coefficients $\mathbf{Y}[\tau, \nu]$ sont rangés par trame $\tau$ puis par fréquence $\nu$.

Trois opérations :

- l'**analyse** $\boldsymbol\Phi^\ast$ : signal $\to$ coefficients (`stft.analysis`) ;
- la **synthèse** $\boldsymbol\Phi$ : coefficients $\to$ signal, adjointe de l'analyse (`stft.synthesis`) ;
- l'**inverse** $\tilde{\boldsymbol\Phi}$ : coefficients $\to$ signal, telle que $\tilde{\boldsymbol\Phi}\boldsymbol\Phi^\ast = \mathrm{Id}$ (`stft.inverse`).

### 2.1 Analyse et synthèse sont adjointes

On munit les signaux du produit scalaire $\langle \mathbf{u}, \mathbf{v}\rangle = \sum_n u[n] v[n]$ et les coefficients du produit scalaire réel $\langle \mathbf{A}, \mathbf{B}\rangle = \mathrm{Re} \sum_{\tau,\nu} \overline{\mathbf{A}[\tau,\nu]}\, \mathbf{B}[\tau,\nu]$. La synthèse est l'adjointe de l'analyse :

$$\langle \boldsymbol\Phi^\ast \mathbf{u}, \mathbf{Z}\rangle = \langle \mathbf{u}, \boldsymbol\Phi \mathbf{Z}\rangle \quad \text{pour tout signal } \mathbf{u} \text{ et tous coefficients } \mathbf{Z}.$$

On le vérifie sur un signal $\mathbf{u}$ et des coefficients $\mathbf{Z}$ tirés au hasard, pour deux fenêtres : Hann, et sinus ajustée (voir plus bas).

In [ ]:
def inner(a, b):
    # Produit scalaire réel, calculé en float64 sur CPU.
    a, b = a.cpu(), b.cpu()
    if a.is_complex():
        return (a.conj() * b).real.double().sum().item()
    return (a.double() * b.double()).sum().item()


def check_adjoint(op, seed=1):
    g = torch.Generator().manual_seed(seed)
    u = torch.randn(op.signal_length, generator=g).to(device)
    Z = torch.randn(op.n_frames, op.n_freqs, dtype=torch.complex64, generator=g).to(device)
    lhs, rhs = inner(op.analysis(u), Z), inner(u, op.synthesis(Z))
    print(f"  <Φ* u, Z> = {lhs:+.6f}   <u, Φ Z> = {rhs:+.6f}   écart relatif = {abs(lhs - rhs) / abs(rhs):.1e}")


for window, tight in (("hann", False), ("sine", True)):
    op = STFT(N, win_length=1024, hop_length=512, window=window, tight=tight)
    print(f"fenêtre {window}, tight={tight}")
    check_adjoint(op)

### 2.2 Trame non ajustée : la synthèse n'est pas l'inverse

Analyse puis synthèse multiplient chaque échantillon par l'enveloppe de la fenêtre :

$$(\boldsymbol\Phi\boldsymbol\Phi^\ast \mathbf{x})[n] = E[n]\, x[n], \qquad E[n] = \sum_\tau w[n - \tau H]^2.$$

L'inverse divise donc la synthèse par $E$ : $\tilde{\boldsymbol\Phi} = (\boldsymbol\Phi\boldsymbol\Phi^\ast)^{-1}\boldsymbol\Phi$. C'est la pseudo-inverse de l'analyse. Avec une fenêtre de Hann et un recouvrement de 50 %, $E$ oscille entre 0,5 et 1 : la synthèse seule déforme le signal.

In [ ]:
stft_hann = STFT(N, win_length=1024, hop_length=512, window="hann", tight=False)
print(stft_hann)

X_hann = stft_hann.analysis(x)
Z = torch.randn(stft_hann.n_frames, stft_hann.n_freqs, dtype=torch.complex64,
                generator=torch.Generator().manual_seed(2)).to(device)
print(f"E[n] dans [{stft_hann.frame_diag.min():.3f}, {stft_hann.frame_diag.max():.3f}]")
print("‖x - Φ Φ* x‖ / ‖x‖  (synthèse)  =", ((x - stft_hann.synthesis(X_hann)).norm() / x.norm()).item())
print("‖x - Φ̃ Φ* x‖ / ‖x‖  (inverse)   =", ((x - stft_hann.inverse(X_hann)).norm() / x.norm()).item())
print("‖Φ Z - Φ̃ Z‖ / ‖Φ̃ Z‖, Z au hasard =",
      ((stft_hann.synthesis(Z) - stft_hann.inverse(Z)).norm() / stft_hann.inverse(Z).norm()).item())

n = np.arange(3 * 1024)
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(n, stft_hann.frame_diag[:n.size])
axes[0].set(title=r"enveloppe $E[n]$, Hann, $H = N_w/2$", xlabel="n", ylim=(0, 1.1))
k = slice(20_000, 20_400)
axes[1].plot(x[k].cpu(), label=r"$x$")
axes[1].plot(stft_hann.synthesis(X_hann)[k].cpu(), "--", label=r"$\Phi\Phi^* x$ (synthèse)")
axes[1].plot(stft_hann.inverse(X_hann)[k].cpu(), ":", label=r"$\tilde\Phi\Phi^* x$ (inverse)")
axes[1].set(title="reconstruction, extrait", xlabel="n"); axes[1].legend(fontsize=8)
plt.tight_layout()

**Remarque : Hann à 50 % n'est pas une trame ajustée.** La fenêtre de Hann s'écrit $w[n] = \sin^2(\pi n / N_w)$. À $H = N_w/2$, la fenêtre décalée vaut $\cos^2$, et la **somme des fenêtres** est constante :

$$w[n] + w[n + N_w/2] = \sin^2 + \cos^2 = 1.$$

C'est la condition COLA (*constant overlap-add*). Elle donne une reconstruction exacte si l'on analyse avec Hann puis additionne les trames **sans les refenêtrer** : la fenêtre de synthèse est alors rectangulaire. Cette synthèse n'est pas l'adjointe de l'analyse.

Une trame ajustée demande plus : la **somme des carrés** doit être constante, car la fenêtre intervient à l'analyse et à la synthèse. Pour Hann à 50 %,

$$E[n] = \sin^4 + \cos^4 = 1 - \tfrac12 \sin^2(2\pi n / N_w) \in [\tfrac12, 1],$$

l'intervalle mesuré ci-dessus.

L'option `tight=True` rend toute fenêtre ajustée en la divisant par $\sqrt{E}$. Pour Hann, le résultat dépend du recouvrement :

- à 75 % ($H = N_w/4$), $E = 3/2$ est constante : la fenêtre ajustée est Hann multipliée par $1/\sqrt{3/2} \approx 0{,}816$ ;
- à 50 %, $E$ varie : la fenêtre ajustée $\sin^2 / \sqrt{\sin^4 + \cos^4}$ n'est plus une fenêtre de Hann. Ce n'est pas non plus la fenêtre sinus, l'autre fenêtre ajustée à 50 %.

In [ ]:
from stft import base_window

hann = base_window("hann", 1024)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(hann, "k", lw=1, label="Hann")
for hop in (512, 256):
    op = STFT(N, win_length=1024, hop_length=hop, window="hann", tight=True)
    ratio = (op.window / hann)[1:]
    print(f"H = {hop} : E[n] dans [{op.frame_diag.min():.7f}, {op.frame_diag.max():.7f}], "
          f"w_ajustée / w_Hann dans [{ratio.min():.4f}, {ratio.max():.4f}]")
    ax.plot(op.window, label=f"Hann ajustée, H = {hop}")
ax.plot(base_window("sine", 1024), "--", label="sinus (ajustée à H = 512)")
ax.set(xlabel="n", title="fenêtres"); ax.legend(fontsize=8)
plt.tight_layout()

### 2.3 Trame ajustée : synthèse = inverse

On divise la fenêtre par $\sqrt{E}$. La fenêtre est alors *ajustée* : $E[n] = 1$ pour tout $n$, donc $\boldsymbol\Phi\boldsymbol\Phi^\ast = \mathrm{Id}$. Deux conséquences :

- la synthèse est l'inverse : $\tilde{\boldsymbol\Phi} = \boldsymbol\Phi$ ;
- l'énergie se conserve, $\lVert\boldsymbol\Phi^\ast\mathbf{x}\rVert^2 = \lVert\mathbf{x}\rVert^2$.

**C'est la transformée utilisée dans la suite** : fenêtre sinus de 1024 échantillons (23 ms), recouvrement de 50 %. La fenêtre sinus est déjà ajustée à ce recouvrement.

In [ ]:
stft = STFT(N, win_length=1024, hop_length=512, n_fft=1024, window="sine", tight=True)
print(stft)

X = stft.analysis(x)          # Φ* x, complex64 de forme (N_τ, N_ν)
Y = stft.analysis(y)          # Φ* y
Z = torch.randn(stft.n_frames, stft.n_freqs, dtype=torch.complex64,
                generator=torch.Generator().manual_seed(2)).to(device)
print(f"E[n] dans [{stft.frame_diag.min():.7f}, {stft.frame_diag.max():.7f}]")
print("‖x - Φ Φ* x‖ / ‖x‖  (synthèse)  =", ((x - stft.synthesis(X)).norm() / x.norm()).item())
print("‖x - Φ̃ Φ* x‖ / ‖x‖  (inverse)   =", ((x - stft.inverse(X)).norm() / x.norm()).item())
print("‖Φ Z - Φ̃ Z‖ / ‖Φ̃ Z‖, Z au hasard =",
      ((stft.synthesis(Z) - stft.inverse(Z)).norm() / stft.inverse(Z).norm()).item())
print("énergie  ‖Φ* x‖ / ‖x‖           =", (X.abs().pow(2).sum().sqrt() / x.norm()).item())

Le spectrogramme montre $|\mathbf{Y}[\tau,\nu]|$ en dB. Le signal est concentré sur peu de coefficients ; le bruit blanc est étalé sur tous.

In [ ]:
def show_spectrogram(Z, title, ax, dynamic_db=80):
    S = 20 * torch.log10(Z.abs().cpu() + 1e-10).numpy().T
    vmax = S.max()
    ax.imshow(S, origin="lower", aspect="auto", cmap="magma", vmin=vmax - dynamic_db, vmax=vmax,
              extent=[0, N / fs, 0, fs / 2000])
    ax.set(title=title, xlabel="temps (s)", ylabel="fréquence (kHz)")


fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
show_spectrogram(X, "signal propre, |Φ* x|", axes[0])
show_spectrogram(Y, f"observation, |Φ* y|, iSNR = {isnr_db:g} dB", axes[1])
plt.tight_layout()

## 3. Seuillages ponctuels

Un seuillage ponctuel multiplie chaque coefficient par un gain qui ne dépend que de son module $r = |\mathbf{Y}[\tau,\nu]|$ :

$$\hat{\mathbf{Y}}[\tau,\nu] = \mathbf{Y}[\tau,\nu]\; h_\theta\big(|\mathbf{Y}[\tau,\nu]|\big).$$

| méthode | gain $h_\theta(r)$ |
|---|---|
| seuillage doux | $(1 - \theta/r)_+$ |
| seuillage dur | $\mathbb{1}\{r > \theta\}$ |
| Wiener empirique | $(1 - \theta^2/r^2)_+$ |
| soustraction spectrale | $(1 - (\theta/r)^\alpha)_+^\beta$ |

Le Wiener empirique est la soustraction spectrale avec $\alpha = 2$, $\beta = 1$.

In [ ]:
r = torch.linspace(0, 4, 400)
theta = 1.0
plt.figure(figsize=(6, 3.5))
plt.plot(r, tfd.soft_gain(r, theta), label="doux")
plt.plot(r, tfd.hard_gain(r, theta), label="dur")
plt.plot(r, tfd.wiener_gain(r, theta), label="Wiener empirique")
plt.plot(r, tfd.spectral_gain(r, theta, alpha=1.0, beta=2.0), "--", label="spectrale, α = 1, β = 2")
plt.axvline(theta, color="gray", lw=0.5)
plt.xlabel("r = |Y| (θ = 1)"); plt.ylabel("gain h_θ(r)"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()

### Le niveau $\lambda \in [0, 1]$

Le seuil ne se règle pas directement. On le paramètre par un niveau $\lambda \in [0, 1]$ :

$$\theta = \lambda\, \max_{\tau,\nu} |\mathbf{Y}[\tau,\nu]|.$$

- $\lambda = 0$ ne touche à rien : $\hat{\mathbf{x}} = \mathbf{y}$.
- $\lambda = 1$ annule tout : $\hat{\mathbf{x}} = 0$.

Le même $\lambda$ a donc le même sens pour tous les signaux et toutes les méthodes. Le niveau de bruit $\sigma$ n'intervient pas dans $g$.

In [ ]:
print("λ = 0 :", f"‖x̂ - y‖ = {(tfd.denoise(y, stft, 'soft', 0.0) - y).norm().item():.1e}")
print("λ = 1 :", f"‖x̂‖     = {tfd.denoise(y, stft, 'soft', 1.0).norm().item():.1e}")

lam = 0.02
for method in ("soft", "hard", "wiener"):
    x_hat = tfd.denoise(y, stft, method, lam)
    print(f"{method:7s} λ = {lam}  oSNR = {snr_db(x, x_hat):6.2f} dB")

### Choix de $\lambda$

On balaie $\lambda$ sur une grille logarithmique de $[10^{-3}, 1]$. Le meilleur $\lambda$ de la grille est le **$\lambda$ oracle** : il utilise le signal propre, c'est une borne haute et non une méthode utilisable. En pratique, on choisit $\lambda$ sans $\mathbf{x}$, par exemple en minimisant l'estimateur SURE du risque (hors de ce notebook).

In [ ]:
lam_grid = np.logspace(-3, 0, 61)


def sweep(method, post=False, **kwargs):
    # oSNR pour chaque λ de la grille.
    return np.array([snr_db(x, tfd.denoise(y, stft, method, lam, sigma=sigma, post=post, **kwargs))
                     for lam in lam_grid])


curves = {m: sweep(m) for m in ("soft", "hard", "wiener")}
curves["spectral α=1, β=2"] = sweep("spectral", alpha=1.0, beta=2.0)

plt.figure(figsize=(7, 4))
for name, c in curves.items():
    k = c.argmax()
    plt.semilogx(lam_grid, c, label=f"{name}  (λ* = {lam_grid[k]:.3f}, {c[k]:.2f} dB)")
plt.axhline(isnr_db, color="gray", ls=":", label="iSNR")
plt.xlabel("λ"); plt.ylabel("oSNR (dB)"); plt.ylim(isnr_db - 5, None)
plt.legend(fontsize=8); plt.grid(alpha=0.3, which="both")
plt.tight_layout()

## 4. Seuillage social : Persistent Empirical Wiener [2, 3]

Un coefficient isolé de grand module est souvent du bruit. Un coefficient entouré de voisins énergiques est plutôt du signal. Le seuillage social en tient compte : le gain dépend de l'énergie voisine

$$\rho[\tau,\nu]^2 = \sum_{\Delta\tau,\Delta\nu} W[\Delta\tau,\Delta\nu]\, |\mathbf{Y}[\tau+\Delta\tau, \nu+\Delta\nu]|^2,$$

avec un noyau $W \geq 0$ de somme 1, de forme (trames, bins). Le PEW applique le gain du Wiener empirique à $\rho$ au lieu de $r$ :

$$\hat{\mathbf{Y}} = \mathbf{Y}\, \big(1 - \theta^2/\rho^2\big)_+, \qquad \theta = \lambda \max_{\tau,\nu} \rho[\tau,\nu].$$

Avec le noyau $1 \times 1$, $\rho = r$ et le PEW est le Wiener empirique. Le Windowed Group Lasso (`"wgl"`) fait de même avec le gain du seuillage doux.

In [ ]:
kernels = {
    "1 × 1 (Wiener)": np.ones((1, 1)),
    "3 × 1 (temps)": np.ones((3, 1)),
    "1 × 3 (fréquence)": np.ones((1, 3)),
    "3 × 3": np.ones((3, 3)),
    "5 × 5": np.ones((5, 5)),
}
pew_curves = {name: sweep("pew", kernel=K) for name, K in kernels.items()}

plt.figure(figsize=(7, 4))
for name, c in pew_curves.items():
    k = c.argmax()
    plt.semilogx(lam_grid, c, label=f"PEW {name}  (λ* = {lam_grid[k]:.3f}, {c[k]:.2f} dB)")
plt.axhline(isnr_db, color="gray", ls=":", label="iSNR")
plt.xlabel("λ"); plt.ylabel("oSNR (dB)"); plt.ylim(isnr_db - 5, None)
plt.legend(fontsize=8); plt.grid(alpha=0.3, which="both")
plt.tight_layout()

Les cartes de gain, au $\lambda$ oracle de chaque méthode, montrent l'effet du voisinage : le PEW garde des régions connexes et supprime les coefficients isolés.

In [ ]:
kernel = kernels["3 × 3"]
lam_w = lam_grid[curves["wiener"].argmax()]
lam_p = lam_grid[pew_curves["3 × 3"].argmax()]

r = Y.abs()
gain_wiener = tfd.wiener_gain(r, lam_w * r.max())
rho = tfd.neighbour_energy(Y, kernel)
gain_pew = tfd.wiener_gain(rho, lam_p * rho.max())

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, G, title in zip(axes, (gain_wiener, gain_pew),
                        (f"gain du Wiener empirique, λ = {lam_w:.3f}", f"gain du PEW 3 × 3, λ = {lam_p:.3f}")):
    ax.imshow(G.cpu().numpy().T, origin="lower", aspect="auto", cmap="gray", vmin=0, vmax=1,
              extent=[0, N / fs, 0, fs / 2000])
    ax.set(title=title, xlabel="temps (s)", ylabel="fréquence (kHz)")
plt.tight_layout()

## 5. Block thresholding de Yu, Mallat et Bacry [1]

Le plan temps-fréquence est découpé en blocs $B_i$ de $N_i$ coefficients. Tous les coefficients d'un bloc reçoivent le même gain

$$a_i = \Big(1 - \frac{\kappa(N_i)}{u_i}\Big)_+, \qquad u_i = \frac{\sum_{B_i} |\mathbf{Y}|^2}{\sum_{B_i} \sigma^2[\tau,\nu]},$$

où $\sigma^2[\tau,\nu] = \mathbb{E}|(\boldsymbol\Phi^\ast\mathbf{n})[\tau,\nu]|^2$ est la variance du bruit du coefficient, et $\kappa(N_i)$ un facteur de sur-soustraction tabulé dans l'article. Dans chaque macrobloc de 8 trames × 16 bins, la forme des blocs est choisie parmi 15 candidates, en minimisant un estimateur de Stein du risque.

La méthode n'a **pas de paramètre libre** : pas de $\lambda$. En revanche, elle demande $\sigma$. La table de $\kappa$ est établie pour une fenêtre sinus à 50 % de recouvrement, la STFT de ce notebook.

In [ ]:
x_yu = tfd.denoise(y, stft, "yu", sigma=sigma)
print(f"Yu et al., sans post-traitement : oSNR = {snr_db(x, x_yu):.2f} dB")

## 6. Post-traitement de Wiener

Yu et al. [1] terminent par une seconde passe. L'estimée $\hat{\mathbf{x}}$ de la première étape sert d'oracle approché dans un filtre de Wiener, appliqué aux coefficients de l'observation :

$$\tilde a[\tau,\nu] = \frac{|(\boldsymbol\Phi^\ast\hat{\mathbf{x}})[\tau,\nu]|^2}{|(\boldsymbol\Phi^\ast\hat{\mathbf{x}})[\tau,\nu]|^2 + \sigma^2[\tau,\nu]}, \qquad \tilde{\mathbf{x}} = \boldsymbol\Phi\big(\tilde a \odot \boldsymbol\Phi^\ast\mathbf{y}\big).$$

Ce post-traitement s'applique à toutes les méthodes (`post=True`) ; il demande $\sigma$. Le $\lambda$ oracle de la chaîne complète n'est pas celui de la première étape : on rebalaie $\lambda$.

In [ ]:
x_yu_post = tfd.denoise(y, stft, "yu", sigma=sigma, post=True)
print(f"Yu et al., avec post-traitement : oSNR = {snr_db(x, x_yu_post):.2f} dB")

methods = {
    "soft": dict(method="soft"),
    "hard": dict(method="hard"),
    "wiener": dict(method="wiener"),
    "PEW 3 × 3": dict(method="pew", kernel=kernels["3 × 3"]),
}
rows = []
for name, kw in methods.items():
    kw = dict(kw)
    method = kw.pop("method")
    c0, c1 = sweep(method, **kw), sweep(method, post=True, **kw)
    rows.append((name, lam_grid[c0.argmax()], c0.max(), lam_grid[c1.argmax()], c1.max()))
rows.append(("Yu et al.", np.nan, snr_db(x, x_yu), np.nan, snr_db(x, x_yu_post)))

print(f"iSNR = {isnr_db:g} dB, λ oracle")
print(f"{'méthode':12s} {'λ*':>7s} {'oSNR':>7s}   {'λ* post':>8s} {'oSNR post':>9s}")
for name, l0, s0, l1, s1 in rows:
    print(f"{name:12s} {l0:7.3f} {s0:7.2f}   {l1:8.3f} {s1:9.2f}")

## Résultat

Spectrogrammes et écoute des meilleures estimées : PEW 3 × 3 au $\lambda$ oracle, et Yu et al., toutes deux avec post-traitement.

In [ ]:
lam_best = rows[3][3]
x_pew_post = tfd.denoise(y, stft, "pew", lam_best, sigma=sigma, kernel=kernels["3 × 3"], post=True)

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, sharey=True)
show_spectrogram(X, "propre", axes[0, 0])
show_spectrogram(Y, f"observation, {snr_db(x, y):.2f} dB", axes[0, 1])
show_spectrogram(stft.analysis(x_pew_post), f"PEW 3 × 3 + post, {snr_db(x, x_pew_post):.2f} dB", axes[1, 0])
show_spectrogram(stft.analysis(x_yu_post), f"Yu et al. + post, {snr_db(x, x_yu_post):.2f} dB", axes[1, 1])
plt.tight_layout()

for name, z in (("observation", y), ("PEW 3 × 3 + post", x_pew_post), ("Yu et al. + post", x_yu_post)):
    print(name)
    display(Audio(z.cpu().numpy(), rate=fs))

## À essayer

- Changer `isnr_db` (0, 5, 20 dB) : comment bougent le $\lambda$ oracle et le classement des méthodes ?
- Changer le tirage `seed` : le $\lambda$ oracle est-il stable ?
- Donner un vrai morceau dans `wav_path`.
- Allonger le noyau du PEW en temps (`np.ones((7, 1))`) ou en fréquence (`np.ones((1, 7))`).
- Essayer le Windowed Group Lasso : `sweep("wgl", kernel=np.ones((3, 3)))`.
- Changer la fenêtre de la STFT (`win_length=2048`, `window="hann"`). Yu et al. sont-ils sensibles à ce choix ?

## Références

1. G. Yu, S. Mallat, E. Bacry, « Audio denoising by time-frequency block thresholding », *IEEE Trans. Signal Processing* 56(5), 2008.
2. K. Siedenburg, M. Dörfler, « Persistent time-frequency shrinkage for audio denoising », *J. Audio Eng. Soc.* 61(1/2), 2013.
3. M. Kowalski, K. Siedenburg, M. Dörfler, « Social sparsity! Neighborhood systems enrich structured shrinkage operators », *IEEE Trans. Signal Processing* 61(10), 2013.